# Validate ONNX int8 Text Encoder vs PyTorch SigLIP2
This notebook validates whether the ONNX int8 text encoder can be used as a drop-in replacement for query encoding

In [ ]:
%pip install -q torch transformers onnxruntime faiss-cpu pillow requests pandas numpy huggingface_hub

In [ ]:
# If needed, uncomment:
# %pip install -q torch transformers onnxruntime faiss-cpu pillow requests pandas numpy huggingface_hub

from __future__ import annotations

import tempfile
import time
from io import BytesIO
from pathlib import Path

import faiss
import numpy as np
import onnxruntime as ort
import pandas as pd
import requests
import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from transformers import AutoModel, AutoProcessor, AutoTokenizer

PT_MODEL_ID = "google/siglip2-base-patch16-naflex"
PT_MODEL_ID_384 = "google/siglip2-base-patch16-384"
ONNX_REPO_ID = "onnx-community/siglip2-base-patch16-384-ONNX"
NUM_MEMES = 15
NUM_PHOTOS = 15
NUM_ILLUSTRATIONS = 15

tmp_root = Path(tempfile.mkdtemp(prefix="validate_onnx_text_encoder_"))
image_dir = tmp_root / "images"
onnx_dir = tmp_root / "onnx"
image_dir.mkdir(parents=True, exist_ok=True)
onnx_dir.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"tmp_root={tmp_root}")
print(f"device={device}")

In [ ]:
def fetch_meme_urls(limit: int) -> list[str]:
    api_url = "https://api.imgflip.com/get_memes"
    r = requests.get(api_url, timeout=20)
    r.raise_for_status()
    payload = r.json()
    if not payload.get("success"):
        raise RuntimeError("imgflip API returned unsuccessful response")
    memes = payload["data"]["memes"]
    return [m["url"] for m in memes[:limit]]


def build_photo_urls(limit: int) -> list[str]:
    return [f"https://picsum.photos/seed/photo-{i}/512/512" for i in range(limit)]


def build_illustration_urls(limit: int) -> list[str]:
    return [
        f"https://placehold.co/512x512/png?text=illustration_{i}&font=roboto"
        for i in range(limit)
    ]


def download_image(url: str, out_path: Path) -> bool:
    try:
        r = requests.get(url, timeout=25)
        r.raise_for_status()
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.save(out_path, format="JPEG", quality=95)
        return True
    except Exception as exc:
        print(f"failed: {url} -> {type(exc).__name__}: {exc}")
        return False


meme_urls = fetch_meme_urls(NUM_MEMES)
photo_urls = build_photo_urls(NUM_PHOTOS)
illustration_urls = build_illustration_urls(NUM_ILLUSTRATIONS)

sources: list[tuple[str, str]] = []
sources.extend([("meme", u) for u in meme_urls])
sources.extend([("photo", u) for u in photo_urls])
sources.extend([("illustration", u) for u in illustration_urls])

records: list[dict] = []
image_id_base = 10_000
for i, (category, url) in enumerate(sources):
    image_id = image_id_base + i
    out_path = image_dir / f"{image_id}.jpg"
    ok = download_image(url, out_path)
    if ok:
        records.append({
            "image_id": image_id,
            "category": category,
            "url": url,
            "path": str(out_path),
        })

# Fallback to extra picsum images if some URLs fail and we drop below 30.
extra_seed = 1000
while len(records) < 30:
    url = f"https://picsum.photos/seed/fallback-{extra_seed}/512/512"
    image_id = image_id_base + len(sources) + extra_seed
    out_path = image_dir / f"{image_id}.jpg"
    if download_image(url, out_path):
        records.append({
            "image_id": image_id,
            "category": "photo_fallback",
            "url": url,
            "path": str(out_path),
        })
    extra_seed += 1
    if extra_seed > 1200 and len(records) < 30:
        raise RuntimeError("Could not download at least 30 images")

if len(records) > 50:
    records = records[:50]

images = [Image.open(r["path"]).convert("RGB") for r in records]
image_ids = [int(r["image_id"]) for r in records]

print(f"downloaded_images={len(records)}")
print(pd.DataFrame(records).groupby("category").size())
pd.DataFrame(records).head()

In [ ]:
processor_naflex = AutoProcessor.from_pretrained(PT_MODEL_ID, trust_remote_code=True)
pt_model_naflex = AutoModel.from_pretrained(PT_MODEL_ID, trust_remote_code=True).to(device)
pt_model_naflex.eval()

processor_384 = AutoProcessor.from_pretrained(PT_MODEL_ID_384, trust_remote_code=True)
pt_model_384 = AutoModel.from_pretrained(PT_MODEL_ID_384, trust_remote_code=True).to(device)
pt_model_384.eval()


def _extract_embedding_tensor(output, kind: str):
    if isinstance(output, torch.Tensor):
        return output

    candidates = ["image_embeds", "pooler_output", "last_hidden_state"] if kind == "image" else ["text_embeds", "pooler_output", "last_hidden_state"]
    for name in candidates:
        if hasattr(output, name):
            val = getattr(output, name)
            if isinstance(val, torch.Tensor):
                if name == "last_hidden_state" and val.ndim == 3:
                    return val[:, 0, :]
                return val

    if isinstance(output, (tuple, list)) and len(output) > 0 and isinstance(output[0], torch.Tensor):
        val = output[0]
        if val.ndim == 3:
            return val[:, 0, :]
        return val

    raise TypeError(f"Unable to extract tensor from output type: {type(output)}")


def encode_images_pytorch(batch_images: list[Image.Image], model, processor, batch_size: int = 8) -> np.ndarray:
    chunks: list[np.ndarray] = []
    for start in range(0, len(batch_images), batch_size):
        b = batch_images[start : start + batch_size]
        inputs = processor(
            images=b,
            return_tensors="pt",
            padding="max_length",
            max_num_patches=256,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            raw = model.get_image_features(**inputs) if hasattr(model, "get_image_features") else model(**inputs)
            feats = _extract_embedding_tensor(raw, kind="image")

        chunks.append(feats.detach().cpu().numpy().astype(np.float32))

    return np.vstack(chunks).astype(np.float32)


def encode_text_pytorch(query: str, model, processor) -> np.ndarray:
    text = query.lower()
    inputs = processor(
        text=[text],
        return_tensors="pt",
        padding="max_length",
        max_length=64,
        truncation=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        raw = model.get_text_features(**inputs) if hasattr(model, "get_text_features") else model(**inputs)
        feats = _extract_embedding_tensor(raw, kind="text")

    arr = feats.detach().cpu().numpy().astype(np.float32)
    faiss.normalize_L2(arr)
    return arr[0]


image_embeddings = encode_images_pytorch(images, model=pt_model_naflex, processor=processor_naflex, batch_size=8)
image_embeddings = image_embeddings.astype(np.float32)
faiss.normalize_L2(image_embeddings)
print("image_embeddings", image_embeddings.shape, image_embeddings.dtype)


In [ ]:
dim = image_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(image_embeddings)

# Production-like mapping: FAISS row index -> image_id
id_mapping: dict[int, int] = {i: image_id for i, image_id in enumerate(image_ids)}


def search_ids(embedding: np.ndarray, k: int = 20) -> list[int]:
    q = embedding.reshape(1, -1).astype(np.float32)
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, k)
    out: list[int] = []
    for idx in idxs[0].tolist():
        if idx < 0:
            continue
        out.append(id_mapping[idx])
    return out


def iou_at_k(ids_a: list[int], ids_b: list[int], k: int) -> float:
    a = set(ids_a[:k])
    b = set(ids_b[:k])
    union = a | b
    if not union:
        return 1.0
    return len(a & b) / len(union)


test_queries = [
    # short
    "cat",
    "dog",
    "meme",
    # medium
    "drake meme template",
    "distracted boyfriend meme",
    "happy person smiling at camera",
    "retro pixel art character",
    # long
    "person looking confused at a computer screen",
    "two people arguing in front of a whiteboard with charts",
    "group of friends laughing together during an outdoor picnic",
    "minimalist illustration of a rocket launching into space",
    # OCR-style
    "stonks graph going up",
    "top text bottom text",
    "breaking news lower third caption",
    "when code works on first try",
    # edge cases
    "a",
    "x",
    "😂",
    "🔥",
    "123",
    "????",
    "",
    # near max-length / truncation stress
    " ".join(["very"] * 70),
    " ".join(["meme", "template", "reaction", "image", "caption"] * 20),
    # practical search-like
    "woman yelling at cat meme",
    "man pointing at presentation slide",
    "simple blue icon style illustration",
]

print(f"faiss_ntotal={index.ntotal}, dim={dim}, queries={len(test_queries)}")

In [ ]:
api = HfApi()
repo_files = api.list_repo_files(ONNX_REPO_ID, repo_type="model")

wanted = []
for f in repo_files:
    if f == "onnx/text_model_int8.onnx":
        wanted.append(f)
    elif f.startswith("tokenizer"):
        wanted.append(f)
    elif f in {"special_tokens_map.json", "added_tokens.json", "vocab.txt", "merges.txt", "spiece.model", "tokenizer_config.json"}:
        wanted.append(f)

if "onnx/text_model_int8.onnx" not in wanted:
    raise FileNotFoundError("onnx/text_model_int8.onnx not found in repo")

for relpath in sorted(set(wanted)):
    hf_hub_download(
        repo_id=ONNX_REPO_ID,
        repo_type="model",
        filename=relpath,
        local_dir=onnx_dir,
        local_dir_use_symlinks=False,
    )

onnx_model_path = onnx_dir / "onnx" / "text_model_int8.onnx"
onnx_tokenizer = AutoTokenizer.from_pretrained(onnx_dir, trust_remote_code=True)

providers = ["CPUExecutionProvider"]
ort_session = ort.InferenceSession(str(onnx_model_path), providers=providers)

ort_input_names = [x.name for x in ort_session.get_inputs()]
ort_output_meta = ort_session.get_outputs()
ort_output_names = [x.name for x in ort_output_meta]
print("ONNX inputs:", ort_input_names)
print("ONNX outputs:", ort_output_names)
for m in ort_output_meta:
    print(f"  output name={m.name}, type={m.type}, shape={m.shape}")

preferred = ["text_embeds", "text_features", "sentence_embedding", "embeddings", "pooler_output", "last_hidden_state"]
selected_onnx_output_name = None
for name in preferred:
    if name in ort_output_names:
        selected_onnx_output_name = name
        break
if selected_onnx_output_name is None:
    selected_onnx_output_name = ort_output_names[0]
print(f"selected_onnx_output_name={selected_onnx_output_name}")

# Tokenizer parity quick check (ONNX tokenizer vs naflex/384 processors)
probe = "drake meme template"
tok_onnx = onnx_tokenizer([probe], padding="max_length", max_length=64, truncation=True, return_tensors="np")
tok_naflex = processor_naflex.tokenizer([probe], padding="max_length", max_length=64, truncation=True, return_tensors="np")
tok_384 = processor_384.tokenizer([probe], padding="max_length", max_length=64, truncation=True, return_tensors="np")

def _tok_diff(a, b):
    if "input_ids" not in a or "input_ids" not in b:
        return None
    x = np.asarray(a["input_ids"])
    y = np.asarray(b["input_ids"])
    if x.shape != y.shape:
        return {"shape_a": x.shape, "shape_b": y.shape}
    return int((x != y).sum())

print("token_id_diffs_vs_onnx:", {
    "naflex": _tok_diff(tok_onnx, tok_naflex),
    "patch16_384": _tok_diff(tok_onnx, tok_384),
})


In [ ]:
def encode_text_onnx(query: str) -> np.ndarray:
    text = query.lower()
    toks = onnx_tokenizer(
        [text],
        return_tensors="np",
        padding="max_length",
        max_length=64,
        truncation=True,
    )

    feed = {}
    for inp in ort_session.get_inputs():
        name = inp.name
        if name in toks:
            arr = toks[name]
        elif name == "token_type_ids":
            arr = np.zeros_like(toks["input_ids"], dtype=np.int64)
        else:
            raise KeyError(f"Tokenizer output missing required ONNX input: {name}")

        if "int64" in inp.type:
            arr = arr.astype(np.int64)
        elif "int32" in inp.type:
            arr = arr.astype(np.int32)
        feed[name] = arr

    outputs = ort_session.run(None, feed)
    out_map = {meta.name: val for meta, val in zip(ort_session.get_outputs(), outputs)}

    chosen = out_map.get(selected_onnx_output_name, outputs[0])
    arr = np.asarray(chosen)
    if arr.ndim == 3:
        arr = arr[:, 0, :]
    elif arr.ndim == 1:
        arr = arr.reshape(1, -1)

    arr = arr.astype(np.float32)
    faiss.normalize_L2(arr)
    return arr[0]


rows = []
for query in test_queries:
    v_naflex = encode_text_pytorch(query, model=pt_model_naflex, processor=processor_naflex)
    v_384 = encode_text_pytorch(query, model=pt_model_384, processor=processor_384)
    v_onnx = encode_text_onnx(query)

    # Retrieval overlap against the same image index (built with naflex image embeddings)
    top_naflex = search_ids(v_naflex, k=20)
    top_384 = search_ids(v_384, k=20)
    top_onnx = search_ids(v_onnx, k=20)

    rows.append({
        "query": query,
        "cos_onnx_vs_naflex": float(np.dot(v_onnx, v_naflex)),
        "l2_onnx_vs_naflex": float(np.linalg.norm(v_onnx - v_naflex)),
        "iou10_onnx_vs_naflex": iou_at_k(top_onnx, top_naflex, k=10),
        "iou20_onnx_vs_naflex": iou_at_k(top_onnx, top_naflex, k=20),

        "cos_onnx_vs_384": float(np.dot(v_onnx, v_384)),
        "l2_onnx_vs_384": float(np.linalg.norm(v_onnx - v_384)),
        "iou10_onnx_vs_384": iou_at_k(top_onnx, top_384, k=10),
        "iou20_onnx_vs_384": iou_at_k(top_onnx, top_384, k=20),

        "cos_naflex_vs_384": float(np.dot(v_naflex, v_384)),
        "l2_naflex_vs_384": float(np.linalg.norm(v_naflex - v_384)),
        "iou20_naflex_vs_384": iou_at_k(top_naflex, top_384, k=20),
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

# Original pass/fail criteria against naflex baseline
mean_cos = float(summary_df["cos_onnx_vs_naflex"].mean())
mean_r20 = float(summary_df["iou20_onnx_vs_naflex"].mean())
verdict = "PASS" if (mean_cos > 0.95 and mean_r20 > 0.90) else "FAIL"

print()
print("Drop-in verdict vs naflex baseline:")
print(f"mean_cosine_sim={mean_cos:.6f}")
print(f"mean_recall@20={mean_r20:.6f}")
print(f"VERDICT={verdict}")

diag_df = pd.DataFrame([
    {"pair": "onnx_vs_naflex", "mean_cos": float(summary_df["cos_onnx_vs_naflex"].mean()), "mean_iou20": float(summary_df["iou20_onnx_vs_naflex"].mean())},
    {"pair": "onnx_vs_384", "mean_cos": float(summary_df["cos_onnx_vs_384"].mean()), "mean_iou20": float(summary_df["iou20_onnx_vs_384"].mean())},
    {"pair": "naflex_vs_384", "mean_cos": float(summary_df["cos_naflex_vs_384"].mean()), "mean_iou20": float(summary_df["iou20_naflex_vs_384"].mean())},
])
print("\nPairwise alignment summary:")
display(diag_df)


In [ ]:
def benchmark_latency(encoder_fn, queries: list[str], n: int = 100, warmup: int = 5) -> dict[str, float]:
    for i in range(warmup):
        _ = encoder_fn(queries[i % len(queries)])

    times_ms = []
    for i in range(n):
        q = queries[i % len(queries)]
        t0 = time.perf_counter()
        _ = encoder_fn(q)
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

    arr = np.array(times_ms, dtype=np.float64)
    return {
        "mean_ms": float(arr.mean()),
        "p50_ms": float(np.percentile(arr, 50)),
        "p99_ms": float(np.percentile(arr, 99)),
    }


naflex_lat = benchmark_latency(lambda q: encode_text_pytorch(q, model=pt_model_naflex, processor=processor_naflex), test_queries, n=100)
pt384_lat = benchmark_latency(lambda q: encode_text_pytorch(q, model=pt_model_384, processor=processor_384), test_queries, n=100)
onnx_lat = benchmark_latency(encode_text_onnx, test_queries, n=100)

latency_df = pd.DataFrame([
    {"encoder": "pytorch_siglip2_naflex", **naflex_lat},
    {"encoder": "pytorch_siglip2_384", **pt384_lat},
    {"encoder": "onnx_int8", **onnx_lat},
])

display(latency_df)
latency_df
